# Lecture 18: Bootstrapping and Covariances

### Mar. 19, 2026

# Agenda

- Introduction (this notebook, *00_Introduction*)
- Linear algebra and matrix reminder
- Bootstrap and jackknife resampling (*01_bootstrapping*)

## The Story So Far: Fitting and Uncertainties

We've built up a toolkit for fitting models to data and estimating uncertainties. Let's remind ourselves what we've done with a simple example: fitting a line $y = mx + b$ to noisy data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fmin

# Generate some noisy linear data
rng = np.random.default_rng(42)
M_TRUE, B_TRUE, SIG = 2.0, 0.8, 1.0
xs = np.linspace(-2, 2, 15)
ys = M_TRUE * xs + B_TRUE + rng.normal(scale=SIG, size=xs.size)

### 1. $\chi^2$ Minimization (Lectures 3 & 5)

We defined $\chi^2 = \sum_i |y_i - f(\mathbf{x}_i, \mathbf{p})|^2 / \sigma_i^2$ and used iterative optimizers like `scipy.optimize.fmin` to find the parameters that minimize it — stepping downhill until convergence. We also saw that `np.polyfit` and `np.linalg.lstsq` solve linear problems directly.

In [ ]:
def chisq(p, xs, ys, sig):
    m, b = p
    return np.sum((ys - (m * xs + b))**2 / sig**2)

m_fit, b_fit = fmin(chisq, [0, 0], args=(xs, ys, SIG))

plt.figure(figsize=(5, 3))
plt.errorbar(xs, ys, yerr=SIG, fmt='k.', label='data')
plt.plot(xs, M_TRUE * xs + B_TRUE, 'b:', label=f'true (m={M_TRUE}, b={B_TRUE})')
plt.plot(xs, m_fit * xs + b_fit, 'r-', label=f'fmin fit (m={m_fit:.2f}, b={b_fit:.2f})')
plt.xlabel('x'); plt.ylabel('y')
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title(r'Iterative $\chi^2$ minimization with fmin')
plt.tight_layout()

### 2. $\Delta\chi^2$ Error Bars (Lecture 3)

We estimated parameter uncertainties by mapping the $\chi^2$ surface and finding where it increases from the minimum. Why the contour levels 1, 4, 9?

Near the minimum, $\chi^2$ is approximately parabolic in each parameter:
$$\chi^2(p) \approx \chi^2_{\min} + \frac{(p - p_{\rm best})^2}{\sigma_p^2}$$

So $\Delta\chi^2 = 1$ means $|p - p_{\rm best}| = 1\sigma_p$, i.e. a $1\sigma$ error bar. More generally, $\Delta\chi^2 = n^2$ gives $n\sigma$:

| $\Delta\chi^2$ | $n\sigma$ |
|:-:|:-:|
| 1 | $1\sigma$ (68%) |
| 4 | $2\sigma$ (95%) |
| 9 | $3\sigma$ (99.7%) |

This requires knowing $\sigma_i$ well, and gets awkward for many correlated parameters.

In [ ]:
chi0 = chisq([m_fit, b_fit], xs, ys, SIG)
ms = np.linspace(m_fit - 0.8, m_fit + 0.8, 100)
bs = np.linspace(b_fit - 0.8, b_fit + 0.8, 100)
M, B = np.meshgrid(ms, bs)
CHI2 = np.array([[chisq([m, b], xs, ys, SIG) for m in ms] for b in bs])

plt.figure(figsize=(5, 4))
plt.contourf(M, B, CHI2 - chi0, levels=np.arange(0, 10, 0.5), cmap='Blues_r')
plt.colorbar(label=r'$\Delta\chi^2$')
plt.contour(M, B, CHI2 - chi0, levels=[1, 4, 9], colors=['r', 'orange', 'yellow'], linewidths=2)
plt.plot(m_fit, b_fit, 'r+', ms=15, mew=2, label='best fit')
plt.plot(M_TRUE, B_TRUE, 'b*', ms=10, label='truth')
plt.xlabel('m'); plt.ylabel('b')
plt.title(r'$\Delta\chi^2$ surface (contours at 1, 4, 9 = 1, 2, 3$\sigma$)')
plt.legend(fontsize=8); plt.tight_layout()

### 3. Bayesian Inference and MCMC (Lectures 5-7)

We reframed fitting as computing a posterior $p(\theta | \text{data}) \propto p(\text{data} | \theta)\, p(\theta)$. MCMC walkers explore this posterior by taking many iterative steps, giving us full probability distributions and parameter correlations. Powerful, but slow and finicky (burn-in, convergence, step sizes, ...). Here's a bare-bones Metropolis-Hastings reminder:

In [ ]:
def log_posterior(p, xs, ys, sig):
    m, b = p
    if not (-10 < m < 10 and -10 < b < 10):  # flat prior
        return -np.inf
    return -0.5 * np.sum((ys - (m * xs + b))**2 / sig**2)

# Metropolis-Hastings sampler
n_steps, step_size = 20_000, 0.05
chain = np.zeros((n_steps, 2))
current = np.array([0.0, 0.0])
for i in range(n_steps):
    proposed = current + rng.normal(scale=step_size, size=2)
    if np.log(rng.uniform()) < log_posterior(proposed, xs, ys, SIG) - log_posterior(current, xs, ys, SIG):
        current = proposed
    chain[i] = current

chain = chain[2000:]  # discard burn-in

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(chain[:, 0], alpha=0.5, lw=0.5); axes[0].set_ylabel('m'); axes[0].set_xlabel('step')
axes[0].axhline(M_TRUE, color='b', ls=':'); axes[0].set_title('MCMC trace')
axes[1].scatter(chain[:, 0], chain[:, 1], s=1, alpha=0.1)
axes[1].plot(M_TRUE, B_TRUE, 'b*', ms=12, label='truth', zorder=5)
axes[1].plot(m_fit, b_fit, 'r+', ms=12, mew=2, label='best fit', zorder=5)
axes[1].set_xlabel('m'); axes[1].set_ylabel('b')
axes[1].set_title('MCMC posterior samples'); axes[1].legend(fontsize=8)
plt.tight_layout()

### What's next?

All of the above methods are **iterative**: they require many function evaluations (fmin), careful exploration of the $\chi^2$ surface ($\Delta\chi^2$), or thousands of MCMC steps. Two questions motivate today's lecture:

- For **linear problems**, do we really need all this iterative machinery? (Spoiler: no — see "The Big(ger) Guns" below.)
- When analytic error bars are **hard to derive** (complicated models, unknown noise properties, correlated errors), is there a simple, nonparametric way to estimate uncertainties? (Spoiler: yes — that's what **bootstrap and jackknife resampling** are for, in *01_bootstrapping*.)

## The Big(ger) Guns

In Lectures 3 and 5, we spent a lot of time with iterative optimizers — `fmin` stepping downhill, `curve_fit` guessing gradients, MCMC walkers exploring posteriors. If you have a linear problem you are trying to solve (or if you can linearize it), you can do *much* better than this iterative mumbo-jumbo.  You can solve it in one shot!  Let's go back to our $y_i = mx_i +b + n_i$ example, but extend it to two dimensions: $z_i = ax_i+by_i+c + n_i$

You know what $x_i$ and $y_i$ are (they are the coordinates of your measurement), and you measured $z_i$.  It turns out you can frame the measurements you made as a matrix multiplication:

\begin{equation}
\begin{pmatrix} 
x_0 & y_0 & 1 \\ 
x_1 & y_1 & 1 \\ 
\vdots & \vdots & \vdots \\ 
x_i & y_i & 1 
\end{pmatrix}
\begin{pmatrix} 
a \\ 
b \\ 
c 
\end{pmatrix}
=
\begin{pmatrix} 
z_0 \\ 
z_1 \\ 
\vdots \\ 
z_i 
\end{pmatrix}
\end{equation}

Let's define the first matrix to be $\mathbf{A}$, the second vector (our parameters to solve for) as $\mathbf{\theta}$, and our measurements $\mathbf{z}$.  Then the above equation reads:
\begin{equation}
\mathbf{A}\cdot\mathbf{\theta} = \mathbf{z}
\end{equation}

### What is $\mathbf{A}^\dagger$?

$\mathbf{A}^\dagger$ is the **conjugate transpose** (also called the Hermitian adjoint) of $\mathbf{A}$: you transpose the matrix and take the complex conjugate of each element. For real-valued data (the common case in this course), this is just the ordinary **transpose** $\mathbf{A}^{\rm T}$: rows become columns and vice versa.

\begin{equation}
\mathbf{A} = \begin{pmatrix} x_0 & y_0 & 1 \\ x_1 & y_1 & 1 \end{pmatrix}
\quad\Rightarrow\quad
\mathbf{A}^\dagger = \mathbf{A}^{\rm T} = \begin{pmatrix} x_0 & x_1 \\ y_0 & y_1 \\ 1 & 1 \end{pmatrix}
\end{equation}

If $\mathbf{A}$ is $N \times M$ (N measurements, M parameters), then $\mathbf{A}^\dagger$ is $M \times N$, and the product $\mathbf{A}^\dagger\mathbf{A}$ is a nice square $M \times M$ matrix that *is* invertible (as long as your measurements aren't degenerate).

In numpy: `A.conj().T` or, for real data, simply `A.T`.

### Solving in one shot

Because $\mathbf{A}$ is not a square matrix, it is not generally invertible, but $\mathbf{A}^\dagger \mathbf{A}$ is.  It will be, in this case, a 3x3 matrix.  This means we can re-write the above as:
\begin{equation}
\mathbf{A}^\dagger\mathbf{A}\cdot\mathbf{\theta} = \mathbf{A}^\dagger\mathbf{z}
\end{equation}
And then, constructing the matrix inverse $(\mathbf{A}^\dagger\mathbf{A})^{-1}$, and applying to both sides, we have:
\begin{equation}
\mathbf{\theta} = (\mathbf{A}^\dagger\mathbf{A})^{-1}\mathbf{A}^\dagger\mathbf{z}
\end{equation}

This is exactly what `np.linalg.lstsq` and `np.polyfit` were doing under the hood in Lecture 5's fitting tutorial — no iteration needed. Let's see it in action on our line-fitting example:

In [ ]:
# Concrete example using our line-fitting data: y = m*x + b
# Design matrix A: each row is [x_i, 1]
A = np.column_stack([xs, np.ones_like(xs)])
print(f"A is {A.shape[0]} x {A.shape[1]} (N_measurements x N_parameters)")
print(f"A[:3] =\n{A[:3]}\n")

# A†  (for real data, just the transpose)
A_dag = A.T
print(f"A† is {A_dag.shape[0]} x {A_dag.shape[1]}")
print(f"A†[:, :3] =\n{A_dag[:, :3]}\n")

# A†A is a square (M x M) matrix we can invert
AtA = A_dag @ A
print(f"A†A = (shape {AtA.shape})\n{AtA}\n")

# One-shot solve: θ = (A†A)⁻¹ A† z
theta = np.linalg.inv(AtA) @ A_dag @ ys
print(f"One-shot solution: m={theta[0]:.4f}, b={theta[1]:.4f}")
print(f"Compare to fmin:   m={m_fit:.4f}, b={b_fit:.4f}")

### Adding noise weighting

The final flourish is, if not all measurements have the same noise, to do inverse-variance weighting.  If we assume our noise for each measurement is independent, we can write down a noise matrix $\mathbf{N}$ that is diagonal and has $\sigma_i^2$ in each row corresponding
to the i$^{\rm th}$ measurement.  Then $\mathbf{N}^{-1}$ is the inverse variance weighting.
Adding that in at the beginning, we can run through the same math to get the final answer:
\begin{equation}
\mathbf{\theta} = (\mathbf{A}^\dagger\mathbf{N}^{-1}\mathbf{A})^{-1}\mathbf{A}^\dagger\mathbf{N}^{-1}\mathbf{z}
\end{equation}

But what about uncertainties? The matrix approach gives us the best-fit parameters in one shot, but we still need error bars. The $\Delta\chi^2$ approach from Lecture 3 works for simple cases, and MCMC from Lectures 6–7 works for complex posteriors, but both have drawbacks. In *01_bootstrapping*, we'll see a complementary approach: **resampling methods** (bootstrap and jackknife) that estimate parameter uncertainties by repeatedly refitting subsets of the data — no assumptions about the noise distribution required.

## Linear Algebra Reminder

$\log p (\{y_n\} \,| \, \theta) = - \frac{1}{2} \sum_{n=1}^{N} [ \frac{[y_n-f_n]^2}{\sigma_n^2} + \log(2\pi \sigma_n^2) ] $

can be extended to more dimensions as:

$\log p (\{y_n\} \,| \, \theta) = - \frac{1}{2} r^{\rm T} C^{-1} r - \frac{1}{2}\log{\rm det} C - \frac{N}{2}\log(2\pi) $

if

$ r= \begin{pmatrix}
y_1-f_1 \\
\vdots \\
y_N-f_N
\end{pmatrix} $

and 

$ C= \begin{pmatrix}
\sigma_1^2 & & 0 \\
& \ddots & \\
0 & & \sigma_N^2
\end{pmatrix} $